# MLflow in Azure Machine Learning


# Notebook Setup

Set project paths and load workspace MLClient.

In [1]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
from pathlib import Path
target = "dp100-learn"
p = Path.cwd()
print(f"Starting working directory: {p}")
while p.name != target and p.parent != p:
    p = p.parent
# Set the path to your project root manually if the above code does not work
p = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn"
# p = "C:/Users/dmika/DEV/Projects-local/dp100-learn"
os.chdir(p)
print("Changed working directory to:", p)
from utils.azureml_utils import *

# Get Azure ML Client based on your environment. Learn more in the tutorials/azureml-first-notebook.ipynb.
ml_client = get_azureml_client()

Starting working directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code
Changed working directory to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn
Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn
Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


Found the config file in: /config.json


In [2]:
import mlflow

# Track model training with MLflow

## Create an MLflow experiment

You'll first create an MLflow experiment. By creating the experiment, you can group all runs within one experiment and make it easier to find the runs in the studio.

In [13]:
experiment_name = "dmdp100-mlflow-basic-tracking"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='', creation_time=1763484884452, experiment_id='2b2fb059-45ac-45ac-9cc0-6b32d6e81d15', last_update_time=None, lifecycle_stage='active', name='dmdp100-mlflow-basic-tracking', tags={}>

## Prepare the data

You'll train a diabetes classification model. The training data is stored in the **data** folder as **diabetes.csv**. 

First, let's read the data:

In [10]:
import pandas as pd

print("Reading data...")

diabetes_df_asset = ml_client.data.get(name="diabetes-data-file", version="1")
df = pd.read_csv(diabetes_df_asset.path)
df.head()

Reading data...


Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,PatientID,Pregnancies,PlasmaGlucose,DiastolicBloodPressure,TricepsThickness,SerumInsulin,BMI,DiabetesPedigree,Age,Diabetic
0,1354778,0,171,80,34,23,43.509726,1.213191,21,0
1,1147438,8,92,93,47,36,21.240576,0.158365,23,0
2,1640031,7,115,47,52,35,41.511523,0.079019,23,0
3,1883350,9,103,78,25,304,29.582192,1.282870,43,1
4,1424119,1,85,59,27,35,42.604536,0.549542,22,0


Next, you'll split the data into features and the label (Diabetes):

In [11]:
print("Splitting data...")
X, y = df[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness','SerumInsulin','BMI','DiabetesPedigree','Age']].values, df['Diabetic'].values

Splitting data...


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

You now have four dataframes:

- `X_train`: The training dataset containing the features.
- `X_test`: The test dataset containing the features.
- `y_train`: The label for the training dataset.
- `y_test`: The label for the test dataset.

You'll use these to train and evaluate the models you'll train.

## Train and track models

To track a model you train, you can use MLflow and enable autologging. The following cell will train a classification model using logistic regression. You'll notice that you don't need to calculate any evaluation metrics because they're automatically created and logged by MLflow.

In [14]:
from sklearn.linear_model import LogisticRegression

with mlflow.start_run(run_name="logistic-regression-diabetes-autolog-1"):
    mlflow.sklearn.autolog()
    model = LogisticRegression(C=1/0.1, solver="liblinear").fit(X_train, y_train)


🏃 View run logistic-regression-diabetes-autolog-1 at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15/runs/deb4bf8d-a880-4366-adc2-907b3781b9c2
🧪 View experiment at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15


You can also use custom logging with MLflow. You can add custom logging to autologging, or you can use only custom logging.

Let's train two more models with scikit-learn. Since you ran the `mlflow.sklearn.autolog()` command before, MLflow will now automatically log any model trained with scikit-learn. To disable the autologging, run the following cell:

In [15]:
mlflow.sklearn.autolog(disable=True)

Now, you can train and track models using only custom logging. 

When you run the following cell, you'll only log one parameter and one metric.

In [16]:
from sklearn.linear_model import LogisticRegression
import numpy as np

with mlflow.start_run(run_name="logistic-regression-diabetes-custom-logging-1"):
    model = LogisticRegression(C=1/0.1, solver="liblinear").fit(X_train, y_train)

    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    mlflow.log_param("regularization_rate", 0.1)
    mlflow.log_metric("Accuracy", acc)

🏃 View run logistic-regression-diabetes-custom-logging-1 at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15/runs/ca7965e7-10a3-422e-b516-0174e4f28d48
🧪 View experiment at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15


The reason why you'd want to track models, could be to compare the results of models you train with different hyperparameter values. 

For example, you just trained a logistic regression model with a regularization rate of 0.1. Now, train another model, but this time with a regularization rate of 0.01. Since you're also tracking the accuracy, you can compare and decide which rate results in a better performing model.

In [17]:
from sklearn.linear_model import LogisticRegression
import numpy as np

with mlflow.start_run(run_name="logistic-regression-diabetes-custom-logging-2"):
    model = LogisticRegression(C=1/0.01, solver="liblinear").fit(X_train, y_train)

    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    mlflow.log_param("regularization_rate", 0.01)
    mlflow.log_metric("Accuracy", acc)

🏃 View run logistic-regression-diabetes-custom-logging-2 at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15/runs/8f3bcb7e-1da4-44a9-b892-b457ef2686c8
🧪 View experiment at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15


Another reason to track your model's results is when you're testing another estimator. All models you've trained so far used the logistic regression estimator. 

Run the following cell to train a model with the decision tree classifier estimator and review whether the accuracy is higher compared to the other runs.

In [18]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np

with mlflow.start_run(run_name="decision-tree-diabetes-custom-logging-1"):
    model = DecisionTreeClassifier().fit(X_train, y_train)

    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    mlflow.log_param("estimator", "DecisionTreeClassifier")
    mlflow.log_metric("Accuracy", acc)

🏃 View run decision-tree-diabetes-custom-logging-1 at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15/runs/be5582d3-263c-4152-a3e3-35f2313da1b4
🧪 View experiment at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15


Finally, let's try to log an artifact. An artifact can be any file. For example, you can plot the ROC curve and store the plot as an image. The image can be logged as an artifact. 

Run the following cell to log a parameter, metric, and an artifact.

In [24]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
import numpy as np

with mlflow.start_run(run_name="decision-tree-diabetes-custom-logging-2"):
    model = DecisionTreeClassifier().fit(X_train, y_train)

    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    # plot ROC curve
    y_scores = model.predict_proba(X_test)

    fpr, tpr, thresholds = roc_curve(y_test, y_scores[:,1])
    fig = plt.figure(figsize=(6, 4))
    # Plot the diagonal 50% line
    plt.plot([0, 1], [0, 1], 'k--')
    # Plot the FPR and TPR achieved by our model
    plt.plot(fpr, tpr)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.savefig("assets/figs/ROC-Curve.png")

    mlflow.log_param("estimator", "DecisionTreeClassifier")
    mlflow.log_metric("Accuracy", acc)
    mlflow.log_artifact("assets/figs/ROC-Curve.png")

🏃 View run decision-tree-diabetes-custom-logging-2 at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/fbd0f4ed-abbc-47e5-b20b-5fb825334ba2/runs/35a2472c-2b9c-459f-bd0a-8576895ae059
🧪 View experiment at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/fbd0f4ed-abbc-47e5-b20b-5fb825334ba2


Review the model's results on the Jobs page of the Azure Machine Learning studio. 

- You'll find the parameters under **Params** in the **Overview** tab.
- You'll find the metrics under **Metrics** in the **Overview** tab, and in the **Metrics** tab.
- You'll find the artifacts in the **Outputs + logs** tab.


# Use MLflow to track Jobs

In [ ]:
experiment_name = "dmdp100-mlflow-job-tracking"
mlflow.set_experiment(experiment_name)
tutorials_materials_src_path = "assets/tutorials-materials/src"

<Experiment: artifact_location='', creation_time=1763485557559, experiment_id='fbd0f4ed-abbc-47e5-b20b-5fb825334ba2', last_update_time=None, lifecycle_stage='active', name='dmdp100-mlflow-job-tracking', tags={}>

## Custom tracking with MLflow

When running a script as a job you can use MLflow in your training script to track the model. MLflow allows you to track any custom parameters, metrics, or artifacts you want to store with your job output.

Run the following cells to create the **train-model-mlflow.py** script in the **src** folder. The script trains a classification model by using the **diabetes.csv** file in the same folder, which is passed as an argument. 

Review the code below to find that the script will import `mlflow` and log:

- The regularization rate as a **parameter**. 
- The accuracy and AUC as **metrics**.
- The plotted ROC curve as an **artifact**.

In [ ]:
%%writefile $tutorials_materials_src_path/train-model-mlflow.py
# import libraries
import mlflow
import argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

def main(args):
    # read data
    df = get_data(args.training_data)

    # split data
    X_train, X_test, y_train, y_test = split_data(df)

    # train model
    model = train_model(args.reg_rate, X_train, X_test, y_train, y_test)

    # evaluate model
    eval_model(model, X_test, y_test)

# function that reads the data
def get_data(path):
    print("Reading data...")
    df = pd.read_csv(path)
    
    return df

# function that splits the data
def split_data(df):
    print("Splitting data...")
    X, y = df[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness',
    'SerumInsulin','BMI','DiabetesPedigree','Age']].values, df['Diabetic'].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

    return X_train, X_test, y_train, y_test

# function that trains the model
def train_model(reg_rate, X_train, X_test, y_train, y_test):
    mlflow.log_param("Regularization rate", reg_rate)
    print("Training model...")
    model = LogisticRegression(C=1/reg_rate, solver="liblinear").fit(X_train, y_train)

    return model

# function that evaluates the model
def eval_model(model, X_test, y_test):
    # calculate accuracy
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    print('Accuracy:', acc)
    mlflow.log_metric("Accuracy", acc)

    # calculate AUC
    y_scores = model.predict_proba(X_test)
    auc = roc_auc_score(y_test,y_scores[:,1])
    print('AUC: ' + str(auc))
    mlflow.log_metric("AUC", auc)

    # plot ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_scores[:,1])
    fig = plt.figure(figsize=(6, 4))
    # Plot the diagonal 50% line
    plt.plot([0, 1], [0, 1], 'k--')
    # Plot the FPR and TPR achieved by our model
    plt.plot(fpr, tpr)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.savefig("ROC-Curve.png")
    mlflow.log_artifact("ROC-Curve.png")    

def parse_args():
    # setup arg parser
    parser = argparse.ArgumentParser()

    # add arguments
    parser.add_argument("--training_data", dest='training_data',
                        type=str)
    parser.add_argument("--reg_rate", dest='reg_rate',
                        type=float, default=0.01)

    # parse args
    args = parser.parse_args()

    # return args
    return args

# run script
if __name__ == "__main__":
    # add space in logs
    print("\n\n")
    print("*" * 60)

    # parse args
    args = parse_args()

    # run main function
    main(args)

    # add space in logs
    print("*" * 60)
    print("\n\n")


Writing utils/tutorials-materials/src/train-model-mlflow-hyper-tuning.py


In [30]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

inputs = {
    "training_data": Input(type=AssetTypes.URI_FILE, path="azureml:diabetes-data-file:1")
    }

In [ ]:
from azure.ai.ml import command

# configure job

job = command(
    code=tutorials_materials_src_path,
    inputs=inputs,
    command="python train-model-mlflow.py --training_data ${{inputs.training_data}}",
    environment="dmdp100env@latest",
    # environment="AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest", # curated environment
    compute="dmdp100-cpu-cluster",
    display_name="diabetes-train-mlflow-cloud-input",
    experiment_name=experiment_name, 
    tags={"model_type": "LogisticRegression"}
    )

# submit job
returned_job = ml_client.create_or_update(job)
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

# Use MLflow to view and search for experiments

The Azure Machine Learning Studio is an easy-to-use UI to view and compare job runs. Alternatively, you can use MLflow to view experiment jobs. 

## Search for experiments

### Show all experiments

In [3]:
experiments = mlflow.search_experiments()
for exp in experiments:
    print(exp.name)

prepare_image
dmdp100-automl-tabular-classification
dmdp100-automl-img-classification
dmdp100-nlp-text-classification-experiment
dmdp100-tracking-playground
dmdp100-mlflow-tracking
dmdp100-mlflow-basic-tracking
dmdp100-mlflow-job-tracking


In [ ]:
from mlflow.entities import ViewType

experiments = mlflow.search_experiments(view_type=ViewType.ALL) # include also archived experiments
for exp in experiments:
    print(exp.name)

prepare_image
dmdp100-automl-tabular-classification
dmdp100-automl-img-classification
dmdp100-nlp-text-classification-experiment
dmdp100-tracking-playground
dmdp100-mlflow-tracking
dmdp100-mlflow-basic-tracking
dmdp100-mlflow-job-tracking


### Retrieve experiment by its name

In [5]:
experiment_name = "dmdp100-mlflow-basic-tracking"
exp = mlflow.get_experiment_by_name(experiment_name)
experiment_id = exp.experiment_id
print(exp)

<Experiment: artifact_location='', creation_time=1763484884452, experiment_id='2b2fb059-45ac-45ac-9cc0-6b32d6e81d15', last_update_time=None, lifecycle_stage='active', name='dmdp100-mlflow-basic-tracking', tags={}>


## View experiment runs

### Show all experiment runs

In [6]:
mlflow.search_runs(experiment_id)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.training_accuracy_score,metrics.training_roc_auc,metrics.training_log_loss,metrics.training_precision_score,...,params.n_jobs,params.dual,params.class_weight,params.regularization_rate,params.estimator,tags.estimator_class,tags.mlflow.runName,tags.estimator_name,tags.mlflow.rootRunId,tags.mlflow.user
0,deb4bf8d-a880-4366-adc2-907b3781b9c2,2b2fb059-45ac-45ac-9cc0-6b32d6e81d15,FINISHED,,2025-11-18 16:59:42.380000+00:00,2025-11-18 17:00:05.790000+00:00,0.790857,0.861986,0.434404,0.78576,...,None,False,None,None,None,sklearn.linear_model._logistic.LogisticRegression,logistic-regression-diabetes-autolog-1,LogisticRegression,deb4bf8d-a880-4366-adc2-907b3781b9c2,Dominik Mika
1,ca7965e7-10a3-422e-b516-0174e4f28d48,2b2fb059-45ac-45ac-9cc0-6b32d6e81d15,FINISHED,,2025-11-18 17:01:08.236000+00:00,2025-11-18 17:01:09.037000+00:00,NaN,NaN,NaN,NaN,...,None,None,None,0.1,None,None,logistic-regression-diabetes-custom-logging-1,None,ca7965e7-10a3-422e-b516-0174e4f28d48,Dominik Mika
2,8f3bcb7e-1da4-44a9-b892-b457ef2686c8,2b2fb059-45ac-45ac-9cc0-6b32d6e81d15,FINISHED,,2025-11-18 17:01:31.236000+00:00,2025-11-18 17:01:32.140000+00:00,NaN,NaN,NaN,NaN,...,None,None,None,0.01,None,None,logistic-regression-diabetes-custom-logging-2,None,8f3bcb7e-1da4-44a9-b892-b457ef2686c8,Dominik Mika
3,be5582d3-263c-4152-a3e3-35f2313da1b4,2b2fb059-45ac-45ac-9cc0-6b32d6e81d15,FINISHED,,2025-11-18 17:01:51.851000+00:00,2025-11-18 17:01:52.575000+00:00,NaN,NaN,NaN,NaN,...,None,None,None,None,DecisionTreeClassifier,None,decision-tree-diabetes-custom-logging-1,None,be5582d3-263c-4152-a3e3-35f2313da1b4,Dominik Mika
4,8372e238-8bb6-4993-a107-75c0a75d71fe,2b2fb059-45ac-45ac-9cc0-6b32d6e81d15,FINISHED,,2025-11-18 17:03:44.240000+00:00,2025-11-18 17:03:45.860000+00:00,NaN,NaN,NaN,NaN,...,None,None,None,None,DecisionTreeClassifier,None,decision-tree-diabetes-custom-logging-2,None,8372e238-8bb6-4993-a107-75c0a75d71fe,Dominik Mika


### Search and order results

In [7]:
mlflow.search_runs(experiment_id, order_by=["start_time DESC"], max_results=2)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.Accuracy,params.estimator,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.rootRunId
0,8372e238-8bb6-4993-a107-75c0a75d71fe,2b2fb059-45ac-45ac-9cc0-6b32d6e81d15,FINISHED,,2025-11-18 17:03:44.240000+00:00,2025-11-18 17:03:45.860000+00:00,0.888667,DecisionTreeClassifier,decision-tree-diabetes-custom-logging-2,Dominik Mika,8372e238-8bb6-4993-a107-75c0a75d71fe
1,be5582d3-263c-4152-a3e3-35f2313da1b4,2b2fb059-45ac-45ac-9cc0-6b32d6e81d15,FINISHED,,2025-11-18 17:01:51.851000+00:00,2025-11-18 17:01:52.575000+00:00,0.888667,DecisionTreeClassifier,decision-tree-diabetes-custom-logging-1,Dominik Mika,be5582d3-263c-4152-a3e3-35f2313da1b4


### Custom filtering

You can even create a query to filter the runs. Filter query strings are written with a simplified version of the SQL `WHERE` clause. 

To filter, you can use two classes of comparators:

- Numeric comparators (metrics): =, !=, >, >=, <, and <=.
- String comparators (params, tags, and attributes): = and !=.

Learn more about [how to track experiments with MLflow](https://learn.microsoft.com/azure/machine-learning/how-to-track-experiments-mlflow).

In [12]:
query = "params.estimator = 'DecisionTreeClassifier'"
mlflow.search_runs(experiment_id, filter_string=query)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.Accuracy,params.estimator,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.rootRunId
0,be5582d3-263c-4152-a3e3-35f2313da1b4,2b2fb059-45ac-45ac-9cc0-6b32d6e81d15,FINISHED,,2025-11-18 17:01:51.851000+00:00,2025-11-18 17:01:52.575000+00:00,0.888667,DecisionTreeClassifier,decision-tree-diabetes-custom-logging-1,Dominik Mika,be5582d3-263c-4152-a3e3-35f2313da1b4
1,8372e238-8bb6-4993-a107-75c0a75d71fe,2b2fb059-45ac-45ac-9cc0-6b32d6e81d15,FINISHED,,2025-11-18 17:03:44.240000+00:00,2025-11-18 17:03:45.860000+00:00,0.888667,DecisionTreeClassifier,decision-tree-diabetes-custom-logging-2,Dominik Mika,8372e238-8bb6-4993-a107-75c0a75d71fe


In [15]:
query = "tags.mlflow.runName = 'decision-tree-diabetes-custom-logging-1'"
mlflow.search_runs(experiment_id, filter_string=query)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.Accuracy,params.estimator,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.rootRunId
0,be5582d3-263c-4152-a3e3-35f2313da1b4,2b2fb059-45ac-45ac-9cc0-6b32d6e81d15,FINISHED,,2025-11-18 17:01:51.851000+00:00,2025-11-18 17:01:52.575000+00:00,0.888667,DecisionTreeClassifier,decision-tree-diabetes-custom-logging-1,Dominik Mika,be5582d3-263c-4152-a3e3-35f2313da1b4


### Search runs in ALL experiments

In [16]:
query = "params.estimator = 'DecisionTreeClassifier'"
mlflow.search_runs(experiment_id, filter_string=query, search_all_experiments=True)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.Accuracy,params.estimator,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.rootRunId
0,be5582d3-263c-4152-a3e3-35f2313da1b4,2b2fb059-45ac-45ac-9cc0-6b32d6e81d15,FINISHED,,2025-11-18 17:01:51.851000+00:00,2025-11-18 17:01:52.575000+00:00,0.888667,DecisionTreeClassifier,decision-tree-diabetes-custom-logging-1,Dominik Mika,be5582d3-263c-4152-a3e3-35f2313da1b4
1,8372e238-8bb6-4993-a107-75c0a75d71fe,2b2fb059-45ac-45ac-9cc0-6b32d6e81d15,FINISHED,,2025-11-18 17:03:44.240000+00:00,2025-11-18 17:03:45.860000+00:00,0.888667,DecisionTreeClassifier,decision-tree-diabetes-custom-logging-2,Dominik Mika,8372e238-8bb6-4993-a107-75c0a75d71fe


# Hyperparameter tuning with MLflow and Azure Machine Learning

There are many machine learning algorithms that require hyperparameters (parameter values that influence training, but can't be determined from the training data itself). For example, when training a logistic regression model, you can use a regularization rate hyperparameter to counteract bias in the model; or when training a convolutional neural network, you can use hyperparameters like learning rate and batch size to control how weights are adjusted and how many data items are processed in a mini-batch respectively. The choice of hyperparameter values can significantly affect the performance of a trained model, or the time taken to train it; and often you need to try multiple combinations to find the optimal solution. 


In [ ]:
experiment_name = "dmdp100-mlflow-hyperparameter-tuning"
mlflow.set_experiment(experiment_name)
tutorials_materials_src_path = "assets/tutorials-materials/src"

## Create the training script
Hyperparameter tuning is ideal when you want to train a machine learning models but vary the input parameters. You'll need to create a training script that expects an input parameter representing one of the algorithm's hyperparameters.

Run the following cells to create the **src** folder and the training script.

Note that the training script expects two input parameters:

- `--training_data` which expects a string. You'll specify the path to a registered data asset as the input training data.
- `--reg_rate` which expects a number, but has a default value of `0.01`. You'll use this input parameter for hyperparameter tuning.

In [37]:
%%writefile $tutorials_materials_src_path/train-model-mlflow-hyper-tuning.py
# import libraries
import mlflow
import argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

def main(args):
    # read data
    df = get_data(args.training_data)

    # split data
    X_train, X_test, y_train, y_test = split_data(df)

    # train model
    model = train_model(args.reg_rate, X_train, X_test, y_train, y_test)

    # evaluate model
    eval_model(model, X_test, y_test)

# function that reads the data
def get_data(path):
    print("Reading data...")
    df = pd.read_csv(path)
    
    return df

# function that splits the data
def split_data(df):
    print("Splitting data...")
    X, y = df[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness',
    'SerumInsulin','BMI','DiabetesPedigree','Age']].values, df['Diabetic'].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

    return X_train, X_test, y_train, y_test

# function that trains the model
def train_model(reg_rate, X_train, X_test, y_train, y_test):
    mlflow.log_param("Regularization rate", reg_rate)
    print("Training model...")
    model = LogisticRegression(C=1/reg_rate, solver="liblinear").fit(X_train, y_train)

    return model

# function that evaluates the model
def eval_model(model, X_test, y_test):
    # calculate accuracy
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    print('Accuracy:', acc)
    mlflow.log_metric("training_accuracy_score", acc)

    # calculate AUC
    y_scores = model.predict_proba(X_test)
    auc = roc_auc_score(y_test,y_scores[:,1])
    print('AUC: ' + str(auc))
    mlflow.log_metric("AUC", auc)

    # plot ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_scores[:,1])
    fig = plt.figure(figsize=(6, 4))
    # Plot the diagonal 50% line
    plt.plot([0, 1], [0, 1], 'k--')
    # Plot the FPR and TPR achieved by our model
    plt.plot(fpr, tpr)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.savefig("ROC-Curve.png")
    mlflow.log_artifact("ROC-Curve.png")    

def parse_args():
    # setup arg parser
    parser = argparse.ArgumentParser()

    # add arguments
    parser.add_argument("--training_data", dest='training_data',
                        type=str)
    parser.add_argument("--reg_rate", dest='reg_rate',
                        type=float, default=0.01)

    # parse args
    args = parser.parse_args()

    # return args
    return args

# run script
if __name__ == "__main__":
    # add space in logs
    print("\n\n")
    print("*" * 60)

    # parse args
    args = parse_args()

    # run main function
    main(args)

    # add space in logs
    print("*" * 60)
    print("\n\n")


Overwriting utils/tutorials-materials/src/train-model-mlflow-hyper-tuning.py


## Configure and run a command job

Run the cell below to train a classification model to predict diabetes. The model is trained by running the **train\.py** script that can be found in the **src** folder. It uses the registered `diabetes-data` data asset as the training data. 

- `code`: specifies the folder that includes the script to run.
- `command`: specifies what to run exactly.
- `environment`: specifies the necessary packages to be installed on the compute before running the command.
- `compute`: specifies the compute to use to run the command.
- `display_name`: the name of the individual job.
- `experiment_name`: the name of the experiment the job belongs to.

Note that the command job only runs the training script once, with a regularization rate of `0.1`. Before you run a sweep job to tune hyperparameters, it's a best practice to test whether your script works as expected with a command job.

In [7]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes

# configure job

job = command(
    code=tutorials_materials_src_path,
    command="python train-model-mlflow-hyper-tuning.py --training_data ${{inputs.diabetes_data}} --reg_rate ${{inputs.reg_rate}}",
    inputs={
        "diabetes_data": Input(type=AssetTypes.URI_FILE, path="azureml:diabetes-data-file:1"),
        "reg_rate": 0.1,
    },
    compute="dmdp100-cpu-cluster",
    environment="dmdp100env@latest",
    # environment="AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest", # curated environment
    display_name="diabetes-train-hyperparameter-tuning-base-job-2",
    experiment_name=experiment_name, 
    tags={"model_type": "LogisticRegression"}
)

# submit job
# returned_job = ml_client.create_or_update(job)
# aml_url = returned_job.studio_url
# print("Monitor your job at", aml_url)

## Define the search space

When your command job has completed successfully, you can configure and run a sweep job. 

First, you'll need to specify the search space for your hyperparameter. To train three models, each with a different regularization rate (`0.01`, `0.1`, or `1`), you can define the search space with a `Choice` hyperparameter. 

In [9]:
from azure.ai.ml.sweep import Choice

command_job_for_sweep = job(
    reg_rate=Choice(values=[0.01, 0.1, 1]),
)

## Configure and submit the sweep job

You'll use the sweep function to do hyperparameter tuning on your training script. To configure a sweep job, you'll need to configure the following:

- `compute`: Name of the compute target to execute the job on.
- `sampling_algorithm`: The hyperparameter sampling algorithm to use over the search space. Allowed values are `random`, `grid` and `bayesian`.
- `primary_metric`: The name of the primary metric reported by each trial job. The metric must be logged in the user's training script using `mlflow.log_metric()` with the same corresponding metric name.
- `goal`: The optimization goal of the `primary_metric`. The allowed values are `maximize` and `minimize`.
- `limits`: Limits for the sweep job. For example, the maximum amount of trials or models you want to train.

Note that the command job is used as the base for the sweep job. The configuration for the command job will be reused by the sweep job.

In [11]:
# apply the sweep parameter to obtain the sweep_job
sweep_job = command_job_for_sweep.sweep(
    compute="dmdp100-cpu-cluster",
    sampling_algorithm="grid",
    primary_metric="training_accuracy_score",
    goal="Maximize",
)

# set the name of the sweep job experiment
sweep_job.experiment_name = experiment_name
sweep_job.display_name = "diabetes-hyperparameter-tuning-sweep-job-1"

# define the limits for this sweep
sweep_job.set_limits(max_total_trials=4, max_concurrent_trials=2, timeout=7200)

Run the following cell to submit the sweep job.

In [13]:
returned_sweep_job = ml_client.create_or_update(sweep_job)
aml_url = returned_sweep_job.studio_url
print("Monitor your job at", aml_url)

Monitor your job at https://ml.azure.com/runs/shy_rocket_cslxnbkm41?wsid=/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw&tid=f78eb1c3-c2e5-404f-bd9e-9f6158703475


## Different custom sweep job configuration

In [ ]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.sweep import Uniform, RandomSamplingAlgorithm, BanditPolicy, MedianStoppingPolicy

# configure job
job = command(
    code=tutorials_materials_src_path,
    command="python train-model-mlflow-hyper-tuning.py --training_data ${{inputs.diabetes_data}} --reg_rate ${{inputs.reg_rate}}",
    inputs={
        "diabetes_data": Input(type=AssetTypes.URI_FILE, path="azureml:diabetes-data-file:1"),
        "reg_rate": 0.1,
    },
    compute="dmdp100-cpu-cluster",
    environment="dmdp100env@latest",
    display_name="diabetes-train-hyperparameter-tuning-base-job-2",
    experiment_name=experiment_name, 
    tags={"model_type": "LogisticRegression"}
)


# define the search space
command_job_for_sweep = job(
    reg_rate=Uniform(0, 1),
)

# define sampling method and apply the sweep parameter to obtain the sweep_job
sweep_job = command_job_for_sweep.sweep(
    compute="dmdp100-cpu-cluster",
    sampling_algorithm = RandomSamplingAlgorithm(seed=2025, rule="sobol"),
    primary_metric="training_accuracy_score",
    goal="Maximize",
)

# define early termination policy
sweep_job.early_termination = BanditPolicy(
    slack_amount = 0.0005, 
    delay_evaluation = 2, 
    evaluation_interval = 1
)
# sweep_job.early_termination = MedianStoppingPolicy(
#     delay_evaluation = 3, 
#     evaluation_interval = 1
# )


# set the name of the sweep job experiment
sweep_job.experiment_name = experiment_name
sweep_job.display_name = "diabetes-hyperparameter-tuning-sweep-job-3"

# define the limits for this sweep
sweep_job.set_limits(max_total_trials=8, max_concurrent_trials=2, timeout=7200)

In [13]:
returned_sweep_job = ml_client.create_or_update(sweep_job)
aml_url = returned_sweep_job.studio_url
print("Monitor your job at", aml_url)

Monitor your job at https://ml.azure.com/runs/coral_nut_8gtwl2djly?wsid=/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw&tid=f78eb1c3-c2e5-404f-bd9e-9f6158703475


# Other

Uncategorized code snippets

In [4]:
import mlflow

mlflow.set_experiment(experiment_name="dmdp100-tracking-playground")

2025/11/18 16:45:43 INFO mlflow.tracking.fluent: Experiment with name 'dmdp100-tracking-playground' does not exist. Creating a new experiment.


<Experiment: artifact_location='', creation_time=1763484342993, experiment_id='c6a82534-f9d7-4ea1-8b3a-6e23776c04a4', last_update_time=None, lifecycle_stage='active', name='dmdp100-tracking-playground', tags={}>

In [3]:
!pip show azureml-mlflow

Name: azureml-mlflow
Version: 1.60.0
Summary: Contains the integration code of AzureML with Mlflow.
Home-page: https://docs.microsoft.com/python/api/overview/azure/ml/?view=azure-ml-py
Author: Microsoft Corp
Author-email: 
License: https://aka.ms/azureml-sdk-license
Location: /anaconda/envs/azureml_py38/lib/python3.10/site-packages
Requires: azure-common, azure-core, azure-identity, azure-mgmt-core, azure-storage-blob, cryptography, jsonpickle, mlflow-skinny, msrest, python-dateutil, pytz
Required-by: azureml-train-automl-runtime
